In [25]:
import os, re, json
# để làm pali-vi, đầu tiên lấy file của pali-vi, ra kết quả (json),
# maping giữa pali-vi và tmc và json của c-pali-tmc-vi (nhớ đổi mn-> mnc)

import unicodedata
# Thư mục chứa các file .md",",
SOURCE_DIR = "../../docs/kinhtieubo/thichminhchau"

# Danh sách tên file .md",", cần xử lý (chỉ tên file, không cần đường dẫn đầy đủ)


FILES = [
{"filename": "kn-013-tap-4-chuong-1-tap-mot-phap.md", "range": (1, 27)},
{"filename": "kn-014-tap-4-chuong-2-tap-hai-phap.md", "range": (28, 49)},
{"filename": "kn-015-tap-4-chuong-3-tap-ba-phap.md", "range": (50, 99)},
{"filename": "kn-016-tap-4-chuong-4-tap-bon-phap.md", "range": (100, 112)},

]

# Heading cấp mấy được coi là "đoạn con" (### = 3, ## = 2, ...)
CHILD_HEADING_LEVEL = 3

NIPATA_TITLES = {
    # "1": "AN 1. Chương Một Pháp",
}


## 2. Các hàm xử lý

In [26]:
TOP_INDEX_RE = re.compile(r'^[a-z]+-0*(\d+)')
H1_RE = re.compile(r'^#\s+(.*)$', re.MULTILINE)
CHILD_RE = re.compile(r'^#{%d}\s+(.*)$' % CHILD_HEADING_LEVEL)
NUM_PREFIX_RE = re.compile(r'^(\d+(?:\.\d+)*)\.?\s*')
SN_PREFIX_RE = re.compile(
    r'^SN\s+(\d+(?:\.\d+)*)\s+',
    re.IGNORECASE
)

from util import slugify

def top_index_from_slug(slug):
    m = TOP_INDEX_RE.match(slug.lower())
    return m.group(1) if m else None


def extract_h1(text):
    m = H1_RE.search(text)
    return m.group(1).strip() if m else None


def insert_path(item, path, default_slug, slug=None, anchor=None):
    """Đi xuống item['children'][...] theo path (list số dạng string), tạo node
    nếu chưa có. Ở node lá: chỉ gắn slug nếu KHÁC trang mặc định của item (tránh
    lặp thừa khi con nằm cùng trang cha)."""
    node = item
    for i, k in enumerate(path):
        node.setdefault("children", {})
        node["children"].setdefault(k, {})
        node = node["children"][k]
        if i == len(path) - 1:
            if slug and slug != default_slug:
                node["slug"] = slug
            if anchor:
                node["anchor"] = anchor


ROMAN_PREFIX_RE = re.compile(r'^\*{0,2}\(([IVXLCDM]+)\)\s*', re.IGNORECASE)

ROMAN_PREFIX_RE = re.compile(
    r'^\*{0,2}(?:\(([IVXLCDM]+)\)|([IVXLCDM]+)\.)\s*',
    re.IGNORECASE
)

_ROMAN_VALUES = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}

def roman_to_int(s):
    s = s.upper()
    total = 0
    prev = 0
    for ch in reversed(s):
        val = _ROMAN_VALUES.get(ch)
        if val is None:
            return None
        total += val if val >= prev else -val
        prev = max(prev, val)
    return total or None

def process_file(dirpath, spec, items, warnings, titles_override):
    filename = spec["filename"]
    file_range = spec.get("range")  # (start, end) hoặc None
    key_override = spec.get("key")  # ghi đè top-index, dùng khi tên file không theo
                                     # đúng quy ước "prefix-SỐ-..." (vd KN: "kn-001-tap-1-...")

    slug = filename[:-3] if filename.endswith(".md") else filename
    filepath = os.path.join(dirpath, filename)
    if not os.path.isfile(filepath):
        warnings.append(f"{filename}: không tìm thấy file, bỏ qua")
        return

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    if key_override is not None:
        key = str(key_override)
    else:
        top_index = top_index_from_slug(slug)
        if top_index is None:
            warnings.append(f"{filename}: không suy ra được số thứ tự (top index) từ tên file, bỏ qua — có thể cần khai \"key\" thủ công")
            return
        key = str(int(top_index))

    is_new_key = key not in items
    item = items.setdefault(key, {})

    if is_new_key:
        if key in titles_override:
            item["title"] = titles_override[key]
        else:
            h1 = extract_h1(text)
            item["title"] = h1 if h1 else f"??? (chưa có tiêu đề cho key {key})"
            if not h1:
                warnings.append(f"{filename}: không tìm thấy H1, cần điền title tay cho key {key}")
        # Trang mặc định khi user chỉ gõ tới top-index (vd "an 1"), không có số con.
        # Với case gộp nhiều file, đây là trang của file ĐẦU TIÊN gặp trong danh sách FILES.
        item["slug"] = slug
    elif key in titles_override and item.get("title") != titles_override[key]:
        item["title"] = titles_override[key]

    default_slug = item["slug"]

    # 1) điền mặc định theo range khai báo -> đảm bảo MỌI số trong range có link,
    #    kể cả khi trong file không có heading riêng cho từng kinh
    if file_range:
        start, end = file_range
        for n in range(start, end + 1):
            insert_path(item, [str(n)], default_slug, slug=slug)

    # 2) quét heading đánh số trong file để bổ sung anchor chính xác (nếu có)
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        # 1. Bắt mọi cấp độ Heading (#, ##, ###, ####...) thay vì dùng CHILD_RE cứng ngắc
        import re
        m_heading = re.match(r'^#+\s+(.*)$', line)
        if m_heading:
            heading_text = m_heading.group(1).strip()
        else:
            # Nếu không có thẻ Heading (#), thử kiểm tra xem có bắt đầu bằng số La Mã không
            if ROMAN_PREFIX_RE.match(line):
                heading_text = line
            else:
                continue

        # Mặc định tạo anchor từ toàn bộ dòng (sẽ bị ghi đè nếu match số La Mã bên dưới)
        anchor = slugify(heading_text)

        num_match = NUM_PREFIX_RE.match(heading_text)
        if num_match:
            numbers = num_match.group(1).split(".")
        else:
            sn_match = SN_PREFIX_RE.match(heading_text)
            if sn_match:
                numbers = sn_match.group(1).split(".")
            else:
                roman_match = ROMAN_PREFIX_RE.match(heading_text)
                if not roman_match:
                    continue

                # Lấy trực tiếp chuỗi La Mã (vd: "I" hoặc "XIV")
                roman = (roman_match.group(1) or roman_match.group(2)).upper()

                # Chuyển đổi La Mã sang số nguyên (I -> 1, XIV -> 14)
                roman_num = roman_to_int(roman)

                if roman_num is None:
                    continue

                # Đặt path key là số (để logic insert_path không bị lỗi)
                numbers = [str(roman_num)]

                # Gán anchor đúng bằng số nguyên đó
                anchor = str(roman_num)

        if len(numbers) == 1:
            path = numbers
        else:
            if numbers[0] != key:
                warnings.append(
                    f'{filename}: heading "{heading_text}" có số đầu ({numbers[0]}) khác key ({key})'
                )
                continue
            path = numbers[1:]

        if not path:
            continue

        # print(anchor) # Mở comment để debug xem anchor đã lấy đúng chưa
        insert_path(item, path, default_slug, slug=slug, anchor=anchor)


## 3. Chạy xử lý

In [27]:
items = {}
warnings = []

for spec in FILES:
    process_file(SOURCE_DIR, spec, items, warnings, NIPATA_TITLES)

if warnings:
    print("⚠️  Cảnh báo:")
    for w in warnings:
        print(" -", w)
else:
    print("Không có cảnh báo.")


Không có cảnh báo.


## 4. Kết quả — dán vào `quicklink-data.js`

Các trường `"???"` là chỗ bạn tự điền (`folder`, edition key, `label`, `path`, `index_length`).

In [29]:
output = {
    'folder': "tinhtieubo",
    'editions': {
        "tmc": {
            "label": "Sujato",
            "path": "sujato-vi",
            "index_length": "2",
            "items": items,
        }
    },
}

print(json.dumps(items, ensure_ascii=False, indent=2))


{
  "13": {
    "title": "Chương Một – Một Pháp",
    "slug": "kn-013-tap-4-chuong-1-tap-mot-phap",
    "children": {
      "1": {
        "anchor": "1"
      },
      "2": {
        "anchor": "2"
      },
      "3": {
        "anchor": "3"
      },
      "4": {
        "anchor": "4"
      },
      "5": {
        "anchor": "5"
      },
      "6": {
        "anchor": "6"
      },
      "7": {
        "anchor": "7"
      },
      "8": {
        "anchor": "8"
      },
      "9": {
        "anchor": "9"
      },
      "10": {
        "anchor": "10"
      },
      "11": {
        "anchor": "11"
      },
      "12": {
        "anchor": "12"
      },
      "13": {
        "anchor": "13"
      },
      "14": {
        "anchor": "14"
      },
      "15": {
        "anchor": "15"
      },
      "16": {
        "anchor": "16"
      },
      "17": {
        "anchor": "17"
      },
      "18": {
        "anchor": "18"
      },
      "19": {
        "anchor": "19"
      },
      "20": {
        "anc

## 5. (Tùy chọn) Ghi ra file JSON

Chạy cell dưới nếu muốn lưu kết quả ra file thay vì chỉ copy từ output ở trên.

In [ ]:
OUT_PATH = "quicklink-data.generated.json"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Đã ghi: {OUT_PATH}")
